# Model 1 — Price Estimation (XGBoost Regression)

**Question:** Given a car's key characteristics, what is its estimated market price in Turkish Lira?

---

## Introduction

This notebook builds a regression model to estimate a car's market price in Turkish Lira (TL) from its key features.

**Key information:**

- **Dataset:** 2,589 rows, 87 columns (unscaled data)
- **Target variable:** `log_Fiyat` — the natural-log-transformed price. After prediction, results are reversed with `np.expm1()` to obtain TL prices.
- **Algorithm:** You **must** use `XGBRegressor`. No other regression algorithm is permitted for this model.
- **Data:** Uses **unscaled** data. Do not use the scaled dataset for this model.

**What you can freely change:**
- Add, remove, or replace features (but keep `log_Fiyat` as the target).
- Tune any hyperparameters of `XGBRegressor`.

**What you cannot change:**
- The general technique category: this must remain a gradient-boosted tree regression (XGBoost).

## 1. Data Import

Load the unscaled dataset and import all required libraries. **This cell is complete — do not modify it.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('../../data/proceed_dataset_without_scaling.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Feature Selection & Train/Test Split

Select your features and split the data into training and test sets.

**TODO:** You may add, remove, or replace features from the recommended set. The only requirement is that `log_Fiyat` remains the target. The recommended set below is a solid starting point.

In [ ]:
# TODO: Select your features. Recommended starting set is shown below.
# You may add, remove, or replace features — but keep log_Fiyat as the target.
recommended_features = [
    'Yıl', 'Kilometre', 'Motor Hacmi', 'Motor Gücü', 'Tork',
    'Hızlanma (0-100)', 'Seri', 'Model', 'Vites Tipi',
    'Kasa Tipi_SUV', 'Çekiş_AWD (Elektronik)',
    'Yakıt Tipi_Dizel', 'Yakıt Tipi_Hibrit',
    'Ortalama Kasko', 'paint_damage_score', 'total_changed_parts'
]
# Filter to columns that actually exist in your dataset
features = [f for f in recommended_features if f in df.columns]
target = 'log_Fiyat'

X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

## 3. Model Training (XGBoost)

Instantiate, fit, and generate predictions with `XGBRegressor`.

**TODO:** Tune the hyperparameters. The values below are starting points only — experiment with `n_estimators`, `max_depth`, `learning_rate`, and `subsample` to improve performance.

In [ ]:
from xgboost import XGBRegressor

# TODO: Tune these hyperparameters. The values below are starting points only.
model = XGBRegressor(
    n_estimators=100,       # TODO: try 200, 500
    max_depth=6,            # TODO: try 3, 4, 8
    learning_rate=0.1,      # TODO: try 0.01, 0.05
    subsample=0.8,          # TODO: try 0.6, 1.0
    random_state=42,
    enable_categorical=True
)

model.fit(X_train, y_train)
y_pred_log = model.predict(X_test)
print("Model training complete.")

## 4. Evaluation — Convert Predictions to TL & Compute Metrics

Convert log-scale predictions back to Turkish Lira using `np.expm1()`, then compute RMSE, MAE, and R².

> **⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs (y_test, y_pred_log, etc.) after training.**

In [ ]:
# ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.
# Convert log-scale predictions back to Turkish Lira
y_test_tl = np.expm1(y_test)
y_pred_tl = np.expm1(y_pred_log)

rmse_tl = np.sqrt(mean_squared_error(y_test_tl, y_pred_tl))
mae_tl = mean_absolute_error(y_test_tl, y_pred_tl)
r2 = r2_score(y_test_tl, y_pred_tl)

print(f"RMSE: {rmse_tl:,.0f} TL")
print(f"MAE:  {mae_tl:,.0f} TL")
print(f"R²:   {r2:.4f}")

## 5. Visualization — Actual vs Predicted Scatter Plot

Plot actual prices against predicted prices. Points near the blue dashed diagonal represent accurate predictions. Colour encodes the absolute prediction error.

> **⚠️ This cell uses placeholder data for illustration. Replace with your actual y_test_tl and y_pred_tl after training.**

In [ ]:
# ⚠️ This cell uses placeholder data for illustration. Replace with your actual y_test_tl and y_pred_tl after training.
residuals_tl = np.abs(y_test_tl - y_pred_tl)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(y_test_tl, y_pred_tl, c=residuals_tl, cmap='RdYlGn_r',
                     alpha=0.6, s=40, edgecolors='none')
plt.colorbar(scatter, ax=ax, label='Absolute Error (TL)')
max_val = max(y_test_tl.max(), y_pred_tl.max())
ax.plot([0, max_val], [0, max_val], 'b--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Price (TL)', fontsize=13)
ax.set_ylabel('Predicted Price (TL)', fontsize=13)
ax.set_title('Actual vs Predicted Car Prices (XGBoost)', fontsize=15)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Visualization — Residual Distribution Histogram

Plot the distribution of prediction errors (predicted − actual). A well-calibrated model should show errors centred around zero.

> **⚠️ Replace residuals_tl with your actual residuals after training.**

In [ ]:
# ⚠️ Replace residuals_tl with your actual residuals after training.
errors_tl = y_pred_tl - y_test_tl
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(errors_tl, bins=50, color='steelblue', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax.set_xlabel('Prediction Error (TL)', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Residual Distribution (Predicted − Actual)', fontsize=15)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Visualization — Feature Importance Bar Chart

Display the top 15 most important features as ranked by XGBoost's built-in importance scores. This helps you understand which features drive predictions most.

> **⚠️ Replace with your actual trained model after training.**

In [ ]:
# ⚠️ Replace with your actual trained model after training.
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind='barh', ax=ax, color='teal')
ax.invert_yaxis()
ax.set_xlabel('XGBoost Importance Score', fontsize=13)
ax.set_title('Top 15 Feature Importances (XGBoost)', fontsize=15)
plt.tight_layout()
plt.show()

## 8. Sample Prediction Table

Display a random sample of 10 test-set cars with their actual prices, predicted prices, and the difference. Green cells indicate over-prediction; red cells indicate under-prediction.

> **⚠️ Replace with your actual test set outputs after training.**

In [ ]:
# ⚠️ Replace with your actual test set outputs after training.
import random
random.seed(42)
idx = random.sample(range(len(X_test)), 10)
sample_df = X_test.iloc[idx].copy()
sample_df['Actual Price (TL)'] = y_test_tl.iloc[idx].values
sample_df['Predicted Price (TL)'] = y_pred_tl[idx]
sample_df['Difference (TL)'] = sample_df['Predicted Price (TL)'] - sample_df['Actual Price (TL)']
display_cols = ['Actual Price (TL)', 'Predicted Price (TL)', 'Difference (TL)']
sample_df[display_cols].style.format({
    'Actual Price (TL)': '{:,.0f}',
    'Predicted Price (TL)': '{:,.0f}',
    'Difference (TL)': '{:+,.0f}'
}).background_gradient(subset=['Difference (TL)'], cmap='RdYlGn_r')

## ⚠️ If Your Model Underperforms

If your model produces poor results (e.g., low R², large RMSE, or predictions far from the diagonal in the scatter plot), **do not discard your results**.

- Keep all outputs as-is.
- In your presentation, document exactly what you observe (metric values, patterns in the residual plot, etc.).
- Write a short hypothesis: Why might the model have underperformed? (e.g., *"The feature set may be missing important categorical variables that explain high-end luxury car prices, causing systematic under-prediction in the upper price range."*)